# Antiparallelogram Linkage — Kinematic Visualization
### Elbow mechanism of the tendon-driven tensegrity manipulator

---

## Background and References

This notebook visualizes the kinematics of the **antiparallelogram four-bar linkage**
used as the **elbow joint** in the FAPS tendon-driven manipulator, as described in:

> **Klein, M.** (2023). *Arbeitsraumanalyse, Simulation und Bewegungsplanung eines
> seilgetriebenen robotischen Manipulators.* Master's thesis, Friedrich-Alexander-
> Universität Erlangen-Nürnberg (FAPS), **Section 3.2.1, Figure 3.5**.

Additional references:

- **Uicker, J. J., Pennock, G. R., & Shigley, J. E.** (2011). *Theory of Machines
  and Mechanisms.* 4th ed., Oxford University Press.
  — Four-bar linkage theory, Kennedy's Theorem (§3.4).
- **McCarthy, J. M., & Soh, G. S.** (2010). *Geometric Design of Linkages.* 2nd ed.,
  Springer. — Closure condition derivation (§1.3).
- **Dijksman, E. A.** (1977). On the Cognates of the Antiparallelogram. *ASME J. Eng.
  for Industry*, 99(3). — Rolling centrode ellipses.

---

## Principle of the Antiparallelogram

An **antiparallelogram** (crossed parallelogram) is a four-bar linkage where:

- Frame and coupler have equal length: $|AB| = |DC| = k_e$
- Both connection rods have equal length: $|AD| = |BC| = l_e$
- The connection rods **cross** each other

The connection rods $AD$ and $BC$ are **purely passive** — they carry no
actuators. Instead, antagonistic **tendons** attached at the moving joints
$D$ and $C$ pull the platform symmetrically in both directions.

```
   A ──────── B        ← static frame / base (top)
    \        /
     X──────X          ← crossing point of rods
    /        \
   D ──────── C        ← moving platform / coupler (bottom)
```

### Rolling Centrode Ellipses

A key kinematic property (Dijksman 1977) is that the **centrodes**
(loci of the instantaneous center of rotation) are **ellipses**:

- **Fixed centrode**: foci at $A$, $B$; center at midpoint of $AB$
- **Moving centrode**: congruent ellipse with foci at $D$, $C$; center at coupler midpoint $M$

Both satisfy $|PA| + |PB| = |PD| + |PC| = l_e$, where $P$ is the ICR.
The semi-axes are $a = l_e/2$, $b = \sqrt{l_e^2 - k_e^2}/2$.
The two **congruent** ellipses **roll on each other** — the contact
point is always the current ICR.

---

## Kinematic Derivation

Frame at $y = 0$, platform at $y < 0$. Fixed pivots:

$$A = \left(-\tfrac{k_e}{2},\, 0\right),\quad B = \left(+\tfrac{k_e}{2},\, 0\right)$$

Angle $\theta$ of rod $AD$ (from vertical, clockwise positive):

$$D = A + l_e \begin{pmatrix} \sin\theta \\ -\cos\theta \end{pmatrix}$$

Angle $\varphi$ of rod $BC$ from the closure condition $|DC| = k_e$
(antiparallelogram branch; McCarthy & Soh 2010, §1.3):

$$\varphi = \theta + 2\,\arctan\!\left(\frac{-k_e \cos\theta}{l_e - k_e \sin\theta}\right)$$

**Symmetric equilibrium** (platform horizontal, centered):
$\theta_0 = \arcsin(k_e / l_e)$.

### Elbow angle vs. rod deviation

The **elbow deflection angle** is the rotation of coupler $DC$ from its
horizontal equilibrium position. Due to the nonlinear closure condition,
this is **not** the same as the rod-angle deviation $\delta = \theta - \theta_0$.
The antiparallelogram amplifies and distorts the angular motion.

The animation sweeps the elbow angle linearly through $\pm 75°$
(Klein 2023, §3.2). For each target elbow angle, the corresponding
rod deviation $\delta$ is found by numerically inverting the closure
condition, ensuring perfectly **symmetric** flexion/extension.

---

## Quick Start

> **Kernel:** requires `numpy`, `scipy`, `plotly`.
> Use the `env_isaaclab` conda environment:
> ```bash
> conda run -n env_isaaclab jupyter notebook
> ```
> Or set the VS Code kernel to `env_isaaclab (Python)`.

1. Run **Cell 2 (Parameters)** — adjust `l_e`, `k_e`, `elbow_max_deg` as needed.
2. Run all remaining cells in order.
3. Click **▶ Play** in the animation output.

In [5]:
# =================================================================
# PARAMETERS — adjust here
# =================================================================

l_e = 150.0      # Rod length [mm] — Klein (2023), §3.2.1
k_e =  60.0      # Joint spacing [mm] — Klein (2023), §3.2.1

# Maximum ELBOW deflection angle [deg].
# Klein (2023), §3.2: elbow joint limits ±75°.
# This is the coupler rotation from equilibrium, NOT the rod-angle
# deviation δ (which differs due to the nonlinear transmission).
elbow_max_deg = 75.0

# Tendon lever arm for force visualization [mm].
lever_arm_mm = 72.5

# Number of animation frames (even number recommended)
n_frames = 120

# =================================================================
# Derived constants — do not modify
# =================================================================
import numpy as np
from scipy.optimize import brentq

assert k_e < l_e, f"Mechanism requires l_e > k_e, but l_e={l_e}, k_e={k_e}"
theta_0 = np.arcsin(k_e / l_e)

# Centrode ellipse parameters: both centrodes are congruent
# with 2a = l_e, foci distance 2c = k_e (Dijksman 1977).
a_ell = l_e / 2.0
c_ell = k_e / 2.0
b_ell = np.sqrt(a_ell**2 - c_ell**2)

# --- Compute rod-angle limits from elbow deflection limit -----------
# The antiparallelogram has a nonlinear transmission: the coupler
# (elbow) rotates faster than the input rod-angle deviation δ, and
# the ratio is different for +δ vs −δ.  We find the δ values that
# produce exactly ±elbow_max elbow deflection.

def _quick_pos(delta):
    """Linkage positions (compact, for delta_max computation only)."""
    theta = theta_0 + delta
    t = -k_e * np.cos(theta) / (l_e - k_e * np.sin(theta))
    phi = theta + 2.0 * np.arctan(t)
    D = np.array([-k_e/2 + l_e*np.sin(theta), -l_e*np.cos(theta)])
    C = np.array([ k_e/2 + l_e*np.sin(phi),   -l_e*np.cos(phi)])
    return D, C

_D0, _C0 = _quick_pos(0.0)
_dc0 = _C0 - _D0  # equilibrium coupler direction = (-k_e, 0)

def _elbow_angle_rad(delta):
    """Signed coupler rotation from equilibrium [rad]."""
    D, C = _quick_pos(delta)
    dc = C - D
    cross = _dc0[0]*dc[1] - _dc0[1]*dc[0]
    dot_  = _dc0[0]*dc[0] + _dc0[1]*dc[1]
    return np.arctan2(cross, dot_)

def delta_for_elbow(target_deg):
    """Rod-angle deviation \u03b4 [rad] for a given elbow deflection [deg].

    Numerically inverts the closure condition.
    """
    target_rad = np.radians(target_deg)
    if abs(target_rad) < 1e-10:
        return 0.0
    if target_rad > 0:
        return brentq(lambda d: _elbow_angle_rad(d) - target_rad,
                       1e-4, np.pi/2 - theta_0 - 0.02)
    else:
        return brentq(lambda d: _elbow_angle_rad(d) - target_rad,
                       -(np.pi/2 + theta_0 - 0.02), -1e-4)

elbow_max_rad = np.radians(elbow_max_deg)
delta_pos = delta_for_elbow(elbow_max_deg)    # \u03b4 for +75\u00b0 elbow
delta_neg = delta_for_elbow(-elbow_max_deg)   # \u03b4 for -75\u00b0 elbow
delta_max = max(abs(delta_pos), abs(delta_neg))  # for centrode range

print(f"l_e            = {l_e:.1f} mm")
print(f"k_e            = {k_e:.1f} mm")
print(f"k_e / l_e      = {k_e/l_e:.4f}")
print(f"theta_0        = {np.degrees(theta_0):.2f}\u00b0 (symmetric equilibrium)")
print(f"elbow_max      = \u00b1{elbow_max_deg:.1f}\u00b0 (coupler rotation limit)")
print(f"\u03b4 for +{elbow_max_deg:.0f}\u00b0 elbow = +{np.degrees(delta_pos):.2f}\u00b0")
print(f"\u03b4 for -{elbow_max_deg:.0f}\u00b0 elbow = {np.degrees(delta_neg):.2f}\u00b0")
print(f"\u03b8 range        = [{np.degrees(theta_0 + delta_neg):.1f}\u00b0, "
      f"{np.degrees(theta_0 + delta_pos):.1f}\u00b0]")
print(f"\nVerification (elbow deflection at limits):")
print(f"  Elbow at +\u03b4: {np.degrees(_elbow_angle_rad(delta_pos)):+.2f}\u00b0")
print(f"  Elbow at -\u03b4: {np.degrees(_elbow_angle_rad(delta_neg)):+.2f}\u00b0")
print(f"\nCentrode ellipse (Dijksman 1977):")
print(f"  a = l_e/2 = {a_ell:.2f} mm")
print(f"  b = \u221a(l_e\u00b2\u2212k_e\u00b2)/2 = {b_ell:.2f} mm")
print(f"  c = k_e/2 = {c_ell:.2f} mm")
print(f"  ICR at equilibrium: (0, {-b_ell:.2f}) mm")

l_e            = 150.0 mm
k_e            = 60.0 mm
k_e / l_e      = 0.4000
theta_0        = 23.58° (symmetric equilibrium)
elbow_max      = ±75.0° (coupler rotation limit)
δ for +75° elbow = +32.42°
δ for -75° elbow = -42.58°
θ range        = [-19.0°, 56.0°]

Verification (elbow deflection at limits):
  Elbow at +δ: +75.00°
  Elbow at -δ: -75.00°

Centrode ellipse (Dijksman 1977):
  a = l_e/2 = 75.00 mm
  b = √(l_e²−k_e²)/2 = 68.74 mm
  c = k_e/2 = 30.00 mm
  ICR at equilibrium: (0, -68.74) mm


In [6]:
# =================================================================
# Kinematics Functions & Trajectory Computation
# =================================================================
import numpy as np


def antiparallel_phi(theta, k_e, l_e):
    """Output rod angle for the antiparallelogram branch.

    Closure condition |DC| = k_e solved via half-angle substitution
    (McCarthy & Soh 2010, \u00a71.3):
        t = tan((phi - theta)/2) = -k_e cos(theta) / (l_e - k_e sin(theta))
    """
    t = -k_e * np.cos(theta) / (l_e - k_e * np.sin(theta))
    return theta + 2.0 * np.arctan(t)


def linkage_positions(delta, k_e, l_e, theta_0):
    """Joint positions at input deviation delta from equilibrium.

    Returns: A, B (fixed), D, C (moving), phi, M (midpoint), theta
    """
    theta = theta_0 + delta
    phi   = antiparallel_phi(theta, k_e, l_e)
    A = np.array([-k_e / 2.0,  0.0])
    B = np.array([ k_e / 2.0,  0.0])
    D = A + l_e * np.array([ np.sin(theta), -np.cos(theta)])
    C = B + l_e * np.array([ np.sin(phi),   -np.cos(phi)])
    M = 0.5 * (D + C)
    return A, B, D, C, phi, M, theta


def instantaneous_center(delta, k_e, l_e, theta_0):
    """ICR via Kennedy's Theorem (Uicker et al. 2011, \u00a73.4).

    Returns None if the extended rods are nearly parallel.
    """
    theta = theta_0 + delta
    phi   = antiparallel_phi(theta, k_e, l_e)
    denom = np.sin(theta - phi)
    if abs(denom) < 1e-9:
        return None
    s = k_e * np.cos(theta) / denom
    return np.array([k_e / 2.0 + s * np.sin(phi), -s * np.cos(phi)])


def elbow_angle(delta, k_e, l_e, theta_0):
    """Signed elbow deflection (coupler rotation from equilibrium) [rad].

    At equilibrium the coupler DC is horizontal; this function returns
    the signed rotation from that reference.
    """
    _, _, D, C, _, _, _ = linkage_positions(delta, k_e, l_e, theta_0)
    dc = C - D
    # Equilibrium coupler direction: (-k_e, 0)
    cross = -k_e * dc[1]
    dot_  = -k_e * dc[0]
    return np.arctan2(cross, dot_)


def centrode_ellipse_pts(a, b, cx=0.0, cy=0.0, angle=0.0, n=200):
    """Points on an axis-aligned ellipse, rotated and translated."""
    t  = np.linspace(0, 2 * np.pi, n)
    ca, sa = np.cos(angle), np.sin(angle)
    ex = a * np.cos(t)
    ey = b * np.sin(t)
    return ca * ex - sa * ey + cx, sa * ex + ca * ey + cy


def moving_centrode_pts(D, C, M, a_ell, b_ell, n=200):
    """Moving centrode ellipse in fixed-frame coordinates."""
    dc = C - D
    angle = np.arctan2(dc[1], dc[0])
    return centrode_ellipse_pts(a_ell, b_ell, M[0], M[1], angle, n)


def tendon_force_dirs(A, B, D, C, arrow_len=35.0):
    """Tendon force direction vectors for the elbow.

    The cables cross \u2014 each tendon pulls from a coupler joint toward
    the opposite-side frame joint (above the attachment point):
      T0: cable at D pulls toward B (crosses to opposite frame pivot)
      T1: cable at C pulls toward A (crosses to opposite frame pivot)
    Returns (t0_base, t0_tip, t1_base, t1_tip).
    """
    dir_t0 = B - D;  dir_t0 = dir_t0 / np.linalg.norm(dir_t0)
    dir_t1 = A - C;  dir_t1 = dir_t1 / np.linalg.norm(dir_t1)
    return D, D + arrow_len * dir_t0, C, C + arrow_len * dir_t1


# --- Precompute trajectories ----------------------------------------
# Use the full operating range (delta_neg to delta_pos) from cell 2.
delta_range = np.linspace(delta_neg, delta_pos, 600)

traj_D = np.array([linkage_positions(d, k_e, l_e, theta_0)[2] for d in delta_range])
traj_C = np.array([linkage_positions(d, k_e, l_e, theta_0)[3] for d in delta_range])
traj_M = np.array([linkage_positions(d, k_e, l_e, theta_0)[5] for d in delta_range])

icr_list  = [instantaneous_center(d, k_e, l_e, theta_0) for d in delta_range]
icr_valid = np.array([p for p in icr_list if p is not None])

# Neutral position & ICR at equilibrium
A0, B0, D0, C0, phi0, M0, th0 = linkage_positions(0.0, k_e, l_e, theta_0)
icr_eq = instantaneous_center(0.0, k_e, l_e, theta_0)

# Fixed centrode ellipse points
fixed_cx, fixed_cy = centrode_ellipse_pts(a_ell, b_ell)

# Moving centrode at neutral
mov_cx_0, mov_cy_0 = moving_centrode_pts(D0, C0, M0, a_ell, b_ell)

# --- Verification ---------------------------------------------------
sums_AB = np.array([
    np.linalg.norm(p - np.array([-k_e/2, 0])) +
    np.linalg.norm(p - np.array([k_e/2, 0]))
    for p in icr_valid])

max_err = max(abs(np.linalg.norm(
    linkage_positions(d, k_e, l_e, theta_0)[2] -
    linkage_positions(d, k_e, l_e, theta_0)[3]) - k_e)
    for d in delta_range)

ea_at_pos = np.degrees(elbow_angle(delta_pos, k_e, l_e, theta_0))
ea_at_neg = np.degrees(elbow_angle(delta_neg, k_e, l_e, theta_0))

print("=== Verification ===")
print(f"Closure |DC|-k_e max error: {max_err:.2e} mm")
print(f"Centrode |PA|+|PB| = {sums_AB.mean():.6f} +/- {sums_AB.std():.2e}  (theory: {l_e})")
print(f"\nElbow deflection symmetry:")
print(f"  At \u03b4_pos: elbow = {ea_at_pos:+.2f}\u00b0")
print(f"  At \u03b4_neg: elbow = {ea_at_neg:+.2f}\u00b0")
print(f"\nNeutral position (\u03b4=0):")
print(f"  D = {D0.round(2)},  C = {C0.round(2)},  M = {M0.round(2)}")
print(f"  ICR = ({icr_eq[0]:.4f}, {icr_eq[1]:.4f})")
print(f"  (= bottom vertex of fixed centrode at (0, -{b_ell:.4f}))")

=== Verification ===
Closure |DC|-k_e max error: 4.26e-14 mm
Centrode |PA|+|PB| = 150.000000 +/- 1.93e-14  (theory: 150.0)

Elbow deflection symmetry:
  At δ_pos: elbow = +75.00°
  At δ_neg: elbow = -75.00°

Neutral position (δ=0):
  D = [  30.   -137.48],  C = [ -30.   -137.48],  M = [   0.   -137.48]
  ICR = (0.0000, -68.7386)
  (= bottom vertex of fixed centrode at (0, -68.7386))


In [7]:
# =================================================================
# Static Overview — Plotly
# =================================================================
import plotly.graph_objects as go
import numpy as np

# --- FAPS color scheme + complements --------------------------------
FAPS_GREEN     = "#97C139"
FAPS_BLUE      = "#296193"
FAPS_GRAY      = "#5F5F5F"
FAPS_LIGHT     = "#E6E6E6"
ROD_BC_COL     = "#C0392B"   # warm red for rod BC
ICR_COLOR      = "#8E44AD"   # purple
TENDON_T0_COL  = "#E74C3C"   # red
TENDON_T1_COL  = "#E67E22"   # orange

# --- 5 poses at symmetric ELBOW angles: -75, -37.5, 0, +37.5, +75 --
show_elbows = [-elbow_max_deg, -elbow_max_deg/2, 0.0,
                elbow_max_deg/2,  elbow_max_deg]
show_deltas = [delta_for_elbow(e) for e in show_elbows]
show_alphas = [0.45, 0.20, 1.0, 0.20, 0.45]

fig = go.Figure()

# Fixed centrode ellipse (full, translucent)
fig.add_trace(go.Scatter(
    x=np.append(fixed_cx, fixed_cx[0]),
    y=np.append(fixed_cy, fixed_cy[0]),
    mode="lines", line=dict(color=FAPS_BLUE, width=2, dash="dash"),
    name="Fixed centrode (foci A, B)", opacity=0.4))

# Moving centrode at neutral (full, translucent)
fig.add_trace(go.Scatter(
    x=np.append(mov_cx_0, mov_cx_0[0]),
    y=np.append(mov_cy_0, mov_cy_0[0]),
    mode="lines", line=dict(color=FAPS_GREEN, width=2, dash="dash"),
    name="Moving centrode (foci D, C)", opacity=0.4))

# ICR path (operating arc on centrode)
fig.add_trace(go.Scatter(
    x=icr_valid[:, 0], y=icr_valid[:, 1],
    mode="markers", marker=dict(size=2, color=ICR_COLOR, opacity=0.25),
    name="ICR path", showlegend=True))

# Linkage ghost positions with tendon force arrows
for i, (dd, ea_deg, alpha) in enumerate(
        zip(show_deltas, show_elbows, show_alphas)):
    A, B, D, C, phi, M, theta = linkage_positions(dd, k_e, l_e, theta_0)
    is_neutral = abs(ea_deg) < 1e-6
    lw_frame   = 6 if is_neutral else 2.5
    lw_rod     = 4 if is_neutral else 2
    lw_coupler = 5 if is_neutral else 2.5
    ms = 10 if is_neutral else 6
    sl = (i == 0)

    fig.add_trace(go.Scatter(
        x=[A[0], B[0]], y=[A[1], B[1]],
        mode="lines", line=dict(color=FAPS_GRAY, width=lw_frame),
        opacity=alpha, showlegend=False))
    fig.add_trace(go.Scatter(
        x=[A[0], D[0]], y=[A[1], D[1]],
        mode="lines", line=dict(color=FAPS_BLUE, width=lw_rod),
        opacity=alpha, showlegend=sl,
        name="Rod AD" if sl else None))
    fig.add_trace(go.Scatter(
        x=[B[0], C[0]], y=[B[1], C[1]],
        mode="lines", line=dict(color=ROD_BC_COL, width=lw_rod),
        opacity=alpha, showlegend=sl,
        name="Rod BC" if sl else None))
    fig.add_trace(go.Scatter(
        x=[D[0], C[0]], y=[D[1], C[1]],
        mode="lines", line=dict(color=FAPS_GREEN, width=lw_coupler),
        opacity=alpha, showlegend=sl,
        name="Coupler DC (platform)" if sl else None))
    fig.add_trace(go.Scatter(
        x=[A[0], B[0], D[0], C[0], M[0]],
        y=[A[1], B[1], D[1], C[1], M[1]],
        mode="markers",
        marker=dict(size=[ms]*2 + [ms]*2 + [ms-2],
                    color=[FAPS_GRAY]*2 + [FAPS_GREEN]*2 + ["#2d5a1e"],
                    line=dict(width=1, color="white")),
        opacity=alpha, showlegend=False))

    # Tendon force direction arrows
    t0b, t0t, t1b, t1t = tendon_force_dirs(A, B, D, C, 35.0)
    sl_t = (i == 0)
    fig.add_trace(go.Scatter(
        x=[t0b[0], t0t[0]], y=[t0b[1], t0t[1]],
        mode="lines+markers",
        line=dict(color=TENDON_T0_COL, width=2.5),
        marker=dict(size=[0, 9], symbol=["circle", "arrow-up"],
                    color=TENDON_T0_COL, angleref="previous"),
        opacity=alpha, showlegend=sl_t,
        name="T\u2080 (D\u2192B)" if sl_t else None))
    fig.add_trace(go.Scatter(
        x=[t1b[0], t1t[0]], y=[t1b[1], t1t[1]],
        mode="lines+markers",
        line=dict(color=TENDON_T1_COL, width=2.5),
        marker=dict(size=[0, 9], symbol=["circle", "arrow-up"],
                    color=TENDON_T1_COL, angleref="previous"),
        opacity=alpha, showlegend=sl_t,
        name="T\u2081 (C\u2192A)" if sl_t else None))

    # Elbow angle annotation on each pose (skip neutral — already labeled)
    if not is_neutral:
        fig.add_annotation(
            x=M[0], y=M[1] - 10,
            text=f"{ea_deg:+.0f}\u00b0",
            showarrow=False,
            font=dict(size=9, color=FAPS_GREEN),
            opacity=alpha)

    if is_neutral:
        for pt, name in [(A, "A"), (B, "B"), (D, "D"), (C, "C"), (M, "M")]:
            ox = -12 if pt[0] < 0 else 12
            oy = 10 if pt[1] >= 0 else -12
            fig.add_annotation(x=pt[0], y=pt[1], text=f"<b>{name}</b>",
                               showarrow=False, xshift=ox, yshift=oy,
                               font=dict(size=13))
        fig.add_annotation(x=0, y=18, text="Static frame (base)",
                           showarrow=False, font=dict(size=10, color=FAPS_GRAY))

# ICR at equilibrium (star)
fig.add_trace(go.Scatter(
    x=[icr_eq[0]], y=[icr_eq[1]],
    mode="markers+text",
    marker=dict(size=16, color=ICR_COLOR, symbol="star",
                line=dict(width=1, color="white")),
    text=[f"  ICR at eq. (0, {icr_eq[1]:.1f})"],
    textposition="middle right", textfont=dict(size=10, color=ICR_COLOR),
    name="ICR at equilibrium"))

# Centrode center (0, 0)
fig.add_trace(go.Scatter(
    x=[0], y=[0],
    mode="markers+text",
    marker=dict(size=8, color=FAPS_BLUE, symbol="x", line=dict(width=2)),
    text=["  Centrode center"], textposition="middle right",
    textfont=dict(size=9, color=FAPS_BLUE),
    name="Centrode center (0, 0)", showlegend=True))

# Base hatching
fig.add_shape(type="rect",
    x0=-k_e/2-15, x1=k_e/2+15, y0=2, y1=12,
    fillcolor=FAPS_LIGHT, line=dict(width=0))
fig.add_shape(type="line",
    x0=-k_e/2-15, x1=k_e/2+15, y0=2, y1=2,
    line=dict(color=FAPS_GRAY, width=2))

# Layout
fig.update_layout(
    title=dict(
        text=(f"Antiparallelogram Elbow \u2014 l<sub>e</sub>={l_e:.0f} mm, "
              f"k<sub>e</sub>={k_e:.0f} mm, "
              f"elbow=\u00b1{elbow_max_deg:.0f}\u00b0"
              f"<br><sup>Klein (2023), \u00a73.2.1, Fig. 3.5</sup>"),
        font=dict(size=14)),
    xaxis=dict(title="x [mm]", scaleanchor="y", scaleratio=1,
               gridcolor="#eee", zeroline=False),
    yaxis=dict(title="y [mm]", gridcolor="#eee", zeroline=False),
    plot_bgcolor="white",
    legend=dict(x=1.02, y=0.5, font=dict(size=10)),
    margin=dict(l=60, r=200, t=80, b=60),
    width=900, height=750)

fig.show()

In [8]:
# =================================================================
# Animation — Plotly (with tendon force direction vectors)
# =================================================================
import plotly.graph_objects as go
import numpy as np

# Sweep ELBOW ANGLE linearly from -elbow_max to +elbow_max and back.
# Convert each target elbow angle to a rod-angle deviation delta.
elbow_half = np.linspace(-elbow_max_deg, elbow_max_deg, n_frames // 2)
elbow_seq  = np.concatenate([elbow_half, elbow_half[::-1]])
delta_seq  = np.array([delta_for_elbow(e) for e in elbow_seq])

ARROW_LEN = 35.0  # tendon arrow length [mm]


def build_frame_data(delta):
    """Compute all dynamic quantities for one animation frame."""
    A, B, D, C, phi, M, theta = linkage_positions(delta, k_e, l_e, theta_0)
    p = instantaneous_center(delta, k_e, l_e, theta_0)
    mx_c, my_c = moving_centrode_pts(D, C, M, a_ell, b_ell, n=150)
    t0b, t0t, t1b, t1t = tendon_force_dirs(A, B, D, C, ARROW_LEN)
    ea = np.degrees(elbow_angle(delta, k_e, l_e, theta_0))
    return dict(A=A, B=B, D=D, C=C, M=M, phi=phi, theta=theta,
                icr=p, mx_c=mx_c, my_c=my_c,
                t0b=t0b, t0t=t0t, t1b=t1b, t1t=t1t, elbow=ea)


ini = build_frame_data(delta_seq[0])

# ===================== STATIC TRACES (0..4) ==========================
traces = []

# 0: Fixed centrode
traces.append(go.Scatter(
    x=np.append(fixed_cx, fixed_cx[0]),
    y=np.append(fixed_cy, fixed_cy[0]),
    mode="lines", line=dict(color=FAPS_BLUE, width=1.5, dash="dash"),
    opacity=0.35, name="Fixed centrode", showlegend=True))

# 1: ICR path
traces.append(go.Scatter(
    x=icr_valid[:, 0], y=icr_valid[:, 1],
    mode="markers", marker=dict(size=2, color=ICR_COLOR, opacity=0.15),
    name="ICR path", showlegend=True))

# 2: Base line
traces.append(go.Scatter(
    x=[-k_e/2-15, k_e/2+15], y=[2, 2],
    mode="lines", line=dict(color=FAPS_GRAY, width=2),
    showlegend=False))

# 3: ICR at equilibrium (star, static reference)
traces.append(go.Scatter(
    x=[icr_eq[0]], y=[icr_eq[1]],
    mode="markers",
    marker=dict(size=12, color=ICR_COLOR, symbol="star", opacity=0.4,
                line=dict(width=1, color="white")),
    name=f"ICR at eq. (0, {icr_eq[1]:.1f})", showlegend=True))

# 4: Centrode center
traces.append(go.Scatter(
    x=[0], y=[0],
    mode="markers",
    marker=dict(size=7, color=FAPS_BLUE, symbol="x", line=dict(width=2)),
    opacity=0.4, name="Centrode center", showlegend=True))

N_STATIC = len(traces)  # = 5

# ===================== DYNAMIC TRACES (5..15) ========================

# 5: Frame AB
traces.append(go.Scatter(
    x=[ini["A"][0], ini["B"][0]], y=[ini["A"][1], ini["B"][1]],
    mode="lines", line=dict(color=FAPS_GRAY, width=6),
    name=f"Frame AB ({k_e:.0f} mm)", showlegend=True))

# 6: Rod AD
traces.append(go.Scatter(
    x=[ini["A"][0], ini["D"][0]], y=[ini["A"][1], ini["D"][1]],
    mode="lines", line=dict(color=FAPS_BLUE, width=3.5),
    name="Rod AD", showlegend=True))

# 7: Rod BC
traces.append(go.Scatter(
    x=[ini["B"][0], ini["C"][0]], y=[ini["B"][1], ini["C"][1]],
    mode="lines", line=dict(color=ROD_BC_COL, width=3.5),
    name="Rod BC", showlegend=True))

# 8: Coupler DC
traces.append(go.Scatter(
    x=[ini["D"][0], ini["C"][0]], y=[ini["D"][1], ini["C"][1]],
    mode="lines", line=dict(color=FAPS_GREEN, width=5),
    name="Coupler DC (platform)", showlegend=True))

# 9: Fixed joints A, B
traces.append(go.Scatter(
    x=[ini["A"][0], ini["B"][0]], y=[ini["A"][1], ini["B"][1]],
    mode="markers",
    marker=dict(size=10, color=FAPS_GRAY, line=dict(width=1, color="white")),
    showlegend=False))

# 10: Moving joints D, C
traces.append(go.Scatter(
    x=[ini["D"][0], ini["C"][0]], y=[ini["D"][1], ini["C"][1]],
    mode="markers",
    marker=dict(size=10, color=FAPS_GREEN, line=dict(width=1, color="white")),
    showlegend=False))

# 11: Platform center M
traces.append(go.Scatter(
    x=[ini["M"][0]], y=[ini["M"][1]],
    mode="markers",
    marker=dict(size=7, color="#2d5a1e", symbol="square"),
    showlegend=False))

# 12: Current ICR
_ix = [ini["icr"][0]] if ini["icr"] is not None else []
_iy = [ini["icr"][1]] if ini["icr"] is not None else []
traces.append(go.Scatter(
    x=_ix, y=_iy,
    mode="markers",
    marker=dict(size=9, color=ICR_COLOR, symbol="circle",
                line=dict(width=1.5, color="white")),
    name="Current ICR", showlegend=True))

# 13: Moving centrode
traces.append(go.Scatter(
    x=np.append(ini["mx_c"], ini["mx_c"][0]),
    y=np.append(ini["my_c"], ini["my_c"][0]),
    mode="lines", line=dict(color=FAPS_GREEN, width=1.5, dash="dash"),
    opacity=0.5, name="Moving centrode", showlegend=True))

# 14: Tendon T0 (D \u2192 B)
traces.append(go.Scatter(
    x=[ini["t0b"][0], ini["t0t"][0]],
    y=[ini["t0b"][1], ini["t0t"][1]],
    mode="lines+markers+text",
    line=dict(color=TENDON_T0_COL, width=3),
    marker=dict(size=[0, 10], symbol=["circle", "arrow-up"],
                color=TENDON_T0_COL, angleref="previous"),
    text=["", "T\u2080"], textposition="top center",
    textfont=dict(size=11, color=TENDON_T0_COL),
    name="T\u2080 (D\u2192B)", showlegend=True))

# 15: Tendon T1 (C \u2192 A)
traces.append(go.Scatter(
    x=[ini["t1b"][0], ini["t1t"][0]],
    y=[ini["t1b"][1], ini["t1t"][1]],
    mode="lines+markers+text",
    line=dict(color=TENDON_T1_COL, width=3),
    marker=dict(size=[0, 10], symbol=["circle", "arrow-up"],
                color=TENDON_T1_COL, angleref="previous"),
    text=["", "T\u2081"], textposition="top center",
    textfont=dict(size=11, color=TENDON_T1_COL),
    name="T\u2081 (C\u2192A)", showlegend=True))

DYN_INDICES = list(range(N_STATIC, len(traces)))  # 5..15

# ===================== BUILD FRAMES ==================================
frames = []
for fi, (delta, ea_deg) in enumerate(zip(delta_seq, elbow_seq)):
    fd = build_frame_data(delta)
    ix = [fd["icr"][0]] if fd["icr"] is not None else []
    iy = [fd["icr"][1]] if fd["icr"] is not None else []

    frame_data = [
        go.Scatter(x=[fd["A"][0], fd["B"][0]],
                   y=[fd["A"][1], fd["B"][1]]),
        go.Scatter(x=[fd["A"][0], fd["D"][0]],
                   y=[fd["A"][1], fd["D"][1]]),
        go.Scatter(x=[fd["B"][0], fd["C"][0]],
                   y=[fd["B"][1], fd["C"][1]]),
        go.Scatter(x=[fd["D"][0], fd["C"][0]],
                   y=[fd["D"][1], fd["C"][1]]),
        go.Scatter(x=[fd["A"][0], fd["B"][0]],
                   y=[fd["A"][1], fd["B"][1]]),
        go.Scatter(x=[fd["D"][0], fd["C"][0]],
                   y=[fd["D"][1], fd["C"][1]]),
        go.Scatter(x=[fd["M"][0]], y=[fd["M"][1]]),
        go.Scatter(x=ix, y=iy),
        go.Scatter(x=np.append(fd["mx_c"], fd["mx_c"][0]),
                   y=np.append(fd["my_c"], fd["my_c"][0])),
        go.Scatter(x=[fd["t0b"][0], fd["t0t"][0]],
                   y=[fd["t0b"][1], fd["t0t"][1]],
                   marker=dict(size=[0, 10],
                               symbol=["circle", "arrow-up"],
                               color=TENDON_T0_COL,
                               angleref="previous")),
        go.Scatter(x=[fd["t1b"][0], fd["t1t"][0]],
                   y=[fd["t1b"][1], fd["t1t"][1]],
                   marker=dict(size=[0, 10],
                               symbol=["circle", "arrow-up"],
                               color=TENDON_T1_COL,
                               angleref="previous")),
    ]

    deg_d = np.degrees(delta)
    deg_t = np.degrees(fd["theta"])
    deg_p = np.degrees(fd["phi"])

    frames.append(go.Frame(
        data=frame_data,
        traces=DYN_INDICES,
        name=str(fi),
        layout=go.Layout(title=dict(text=(
            f"Antiparallelogram Elbow \u2014 "
            f"l<sub>e</sub>={l_e:.0f}, k<sub>e</sub>={k_e:.0f} mm<br>"
            f"<sup>elbow={fd['elbow']:+.1f}\u00b0  |  "
            f"\u03b4={deg_d:+.1f}\u00b0  |  "
            f"\u03b8={deg_t:+.1f}\u00b0  |  "
            f"\u03c6={deg_p:+.1f}\u00b0</sup>")))))

# ===================== SLIDER ========================================
slider_steps = []
step_interval = max(1, len(elbow_seq) // 30)
for fi in range(0, len(elbow_seq), step_interval):
    slider_steps.append(dict(
        args=[[str(fi)], dict(frame=dict(duration=0, redraw=True),
                               mode="immediate")],
        label=f"{elbow_seq[fi]:+.0f}\u00b0",
        method="animate"))

# ===================== ASSEMBLE ======================================
fig_a = go.Figure(data=traces, frames=frames)

fig_a.update_layout(
    title=dict(
        text=(f"Antiparallelogram Elbow \u2014 "
              f"l<sub>e</sub>={l_e:.0f}, k<sub>e</sub>={k_e:.0f} mm<br>"
              f"<sup>Klein (2023), \u00a73.2.1</sup>"),
        font=dict(size=14)),
    xaxis=dict(title="x [mm]", scaleanchor="y", scaleratio=1,
               range=[-180, 180], gridcolor="#eee", zeroline=False),
    yaxis=dict(title="y [mm]",
               range=[-190, 50], gridcolor="#eee", zeroline=False),
    plot_bgcolor="white",
    legend=dict(x=1.02, y=0.5, font=dict(size=9)),
    margin=dict(l=60, r=220, t=80, b=80),
    width=950, height=750,
    updatemenus=[dict(
        type="buttons", x=0.08, y=-0.06, xanchor="left",
        buttons=[
            dict(label="\u25b6 Play", method="animate",
                 args=[None, dict(frame=dict(duration=40, redraw=True),
                                  fromcurrent=True,
                                  transition=dict(duration=0))]),
            dict(label="\u23f8 Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ])],
    sliders=[dict(
        active=0, x=0.08, len=0.84,
        xanchor="left", y=-0.02,
        currentvalue=dict(prefix="elbow = ", suffix="\u00b0",
                          font=dict(size=12)),
        steps=slider_steps)])

fig_a.add_shape(type="rect",
    x0=-k_e/2-15, x1=k_e/2+15, y0=2, y1=12,
    fillcolor=FAPS_LIGHT, line=dict(width=0))

fig_a.show()
print("Use the Play button or drag the slider to animate the mechanism.")

Use the Play button or drag the slider to animate the mechanism.


---

## Summary

| Property | Value |
|---|---|
| Rod length $l_e$ | **150 mm** |
| Joint spacing $k_e$ | **60 mm** |
| Symmetric equilibrium $\theta_0$ | $\arcsin(60/150) \approx 23.6°$ |
| Elbow deflection range | $\pm 75°$ (Klein 2023, §3.2) |
| Rod deviation $\delta$ for $+75°$ | computed numerically (~32°) |
| Rod deviation $\delta$ for $-75°$ | computed numerically (~43°) |
| Centrode semi-axes | $a = 75$ mm, $b \approx 68.7$ mm |
| ICR at equilibrium | $(0,\; -68.7)$ mm |
| Tendon lever arm | $\pm 72.5$ mm (Klein 2023) |

### Rolling Centrode Ellipses

The instantaneous center of rotation (ICR) traces an arc on the
**fixed centrode** — an ellipse with foci at the frame pivots $A$, $B$.
The **moving centrode** is a congruent ellipse with foci at the coupler
endpoints $D$, $C$. Both satisfy $|PA|+|PB| = |PD|+|PC| = l_e$
(Dijksman 1977). The two ellipses roll on each other without slipping.

### Nonlinear Transmission

The antiparallelogram amplifies angular motion: a small rod deviation
$\delta$ produces a larger coupler (elbow) rotation. The transmission
ratio is not constant, so the rod deviations for $+75°$ and $-75°$
elbow angle differ in magnitude. The plots parametrize by **elbow angle**
(symmetric $\pm 75°$) rather than rod deviation $\delta$, ensuring the
depicted poses show equal deflection in both directions.

### Tendon Force Directions

The connection rods $AD$ and $BC$ are purely passive. The elbow is
actuated by two antagonistic tendons (Klein 2023, §3.2.1):
- **T₀**: attached at $D$, pulls toward $B$ (crossing cable) — positive torque
- **T₁**: attached at $C$, pulls toward $A$ (crossing cable) — negative torque

In the IsaacLab tendon model, these map to:
$\tau_{\mathrm{elbow}} = 0.0725 \cdot T_0 - 0.0725 \cdot T_1$

---

## References

1. **Klein, M.** (2023). *Arbeitsraumanalyse, Simulation und Bewegungsplanung
   eines seilgetriebenen robotischen Manipulators.* Master's thesis, FAU
   Erlangen-Nürnberg (FAPS). §3.2, **§3.2.1**, Fig. 3.5.

2. **Uicker, J. J., Pennock, G. R., & Shigley, J. E.** (2011). *Theory of
   Machines and Mechanisms.* 4th ed., Oxford University Press.
   (Ch. 2: four-bar linkage; §3.4: ICR, Kennedy's Theorem)

3. **McCarthy, J. M., & Soh, G. S.** (2010). *Geometric Design of Linkages.*
   2nd ed., Springer. (§1.3: closure condition)

4. **Dijksman, E. A.** (1977). On the Cognates of the Antiparallelogram.
   *ASME J. Eng. for Industry*, 99(3). (Rolling centrode ellipses)